In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# Start Spark session
spark = SparkSession.builder \
    .appName("KafkaTripsStreaming") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1") \
    .getOrCreate()

# Define schema of your JSON messages
trip_schema = StructType([
    StructField("pickup_datetime", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("passenger_count", IntegerType(), True)
])

# Read streaming data from Kafka
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "trips") \
    .option("startingOffsets", "earliest") \
    .load()

# Convert value from binary to string and parse JSON
trips = df.selectExpr("CAST(value AS STRING) as json") \
          .select(from_json(col("json"), trip_schema).alias("data")) \
          .select("data.*")

trips.printSchema()

root
 |-- pickup_datetime: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)



In [ ]:
spark.sparkContext.setLogLevel("WARN")

In [ ]:
from pyspark.sql.functions import window

agg = trips.withColumn("pickup_datetime", col("pickup_datetime").cast(TimestampType())) \
    .withWatermark("pickup_datetime", "2 minutes") \
    .groupBy(
        window(col("pickup_datetime"), "1 minute"),
        col("PULocationID")
    ).count()

# Write streaming output to console
query = agg.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", "/home/jovyan/work/output_parquet") \
    .option("checkpointLocation", "/home/jovyan/work/checkpoint") \
    .start()

query.awaitTermination()

In [ ]:
spark.read.parquet("/home/jovyan/work/output_parquet").show(10, truncate=False)

In [ ]:
query.isActive  # should be True

In [ ]:
query.status

In [ ]:
query.stop()

In [ ]:
spark.catalog.listTables()